# Duka Electronics — H1 2026 Sales Exploration

## Executive Summary
- **Nairobi** is the top branch with KES 26,106,100 (36.4% of the KES 71,792,500 total revenue across 1,000 orders).
- **Laptop** is the top product by revenue at KES 41,535,000 (57.9% of revenue), while **Tablet** leads on units with 691 sold.
- Weekend orders are 27.7% of orders (277), averaging KES 71,972 per order vs KES 71,724 on weekdays.
- **April** was the best month (KES 13,518,300) and **June** the worst (KES 10,903,400), a 24.0% gap.
- 33 Premium online orders from Nairobi and Mombasa generated KES 5,850,000, which is 8.1% of revenue.

## Task 1 · Load & first-five-minutes inspection

In [1]:
import pandas as pd, numpy as np
pd.options.display.float_format = "{:,.2f}".format
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("shape:", df.shape); print(df.dtypes); print("\nmissing values:\n", df.isna().sum())
print("\ndate range:", df.order_date.min().date(), "to", df.order_date.max().date())
for c in ["branch","product","channel"]: print("\n", df[c].value_counts().to_dict())
print("duplicate order_id:", df.order_id.duplicated().sum())
df.describe()

shape: (1000, 7)
order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

missing values:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

date range: 2026-01-01 to 2026-06-29

 {'Nairobi': 376, 'Mombasa': 186, 'Nakuru': 162, 'Kisumu': 159, 'Eldoret': 117}

 {'Laptop': 220, 'Tablet': 215, 'Phone': 194, 'Headphones': 187, 'Charger': 184}

 {'In-store': 579, 'Online': 320, 'Phone order': 101}
duplicate order_id: 0


,order_id,order_date,quantity,unit_price
count,"1,000.00",1000,"1,000.00","1,000.00"
mean,"1,500.50",2026-03-30 08:44:09.600000,3.01,"23,918.30"
min,"1,001.00",2026-01-01 00:00:00,1.00,"1,200.00"
25%,"1,250.75",2026-02-14 00:00:00,2.00,"2,500.00"
50%,"1,500.50",2026-03-29 00:00:00,3.00,"15,000.00"
75%,"1,750.25",2026-05-14 00:00:00,4.00,"28,000.00"
max,"2,000.00",2026-06-29 00:00:00,5.00,"65,000.00"
std,288.82,NaN,1.42,"23,908.79"


**Inspection notes (what the first five minutes showed).**
- **Shape and types:** 1,000 rows × 7 columns; `order_date` was parsed to datetime on load (range 2026-01-01 → 2026-06-29), the rest are numbers or short text categories.
- **Completeness and uniqueness:** 0 missing values and 0 repeated `order_id`, so no cleaning was needed before analysis.
- **Categories:** 5 branches, 5 products and 3 channels (In-store, Online, Phone order), with no spelling variants.
- **Plausibility:** each product has exactly one `unit_price`; `quantity` runs from 1 to 5; orders appear on 179 of the 180 calendar days, so the period is covered evenly.
- **Implication:** revenue can be derived directly as `quantity × unit_price`, and monthly comparisons are fair except for the last month, which is cut short on 2026-06-29.


## Task 2 · Derived columns

In [2]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = pd.cut(df["unit_price"], bins=[0, 5000, 30000, np.inf], right=False,
                          labels=["Budget", "Mid", "Premium"]).astype(str)
df[["order_id","revenue","month","weekday","is_weekend","price_band"]].head()

,order_id,revenue,month,weekday,is_weekend,price_band
0,1001,45000,June,Wednesday,False,Mid
1,1002,260000,February,Monday,False,Premium
2,1003,140000,January,Monday,False,Mid
3,1004,12500,April,Sunday,True,Budget
4,1005,7500,March,Saturday,True,Budget


`revenue = quantity × unit_price`. `price_band` splits `unit_price` into **Budget** (below KES 5,000), **Mid** (KES 5,000 to 29,999) and **Premium** (KES 30,000 and above); Premium here means: Laptop.

## Task 3 · Branch ranking: revenue vs orders

In [3]:
b = df.groupby("branch").agg(total_revenue=("revenue","sum"), orders=("order_id","count"))
b["avg_order_value"] = b.total_revenue/b.orders
b["rank_revenue"] = b.total_revenue.rank(ascending=False).astype(int)
b["rank_orders"] = b.orders.rank(ascending=False, method="min").astype(int)
b.sort_values("total_revenue", ascending=False)

,total_revenue,orders,avg_order_value,rank_revenue,rank_orders
branch,,,,,
Nairobi,26106100,376,"69,431.12",1,1
Mombasa,14003700,186,"75,288.71",2,2
Nakuru,11970400,162,"73,891.36",3,3
Kisumu,10868100,159,"68,352.83",4,4
Eldoret,8844200,117,"75,591.45",5,5


**Answer.** By revenue: Nairobi > Mombasa > Nakuru > Kisumu > Eldoret. By orders: Nairobi > Mombasa > Nakuru > Kisumu > Eldoret. The two rankings are identical: order volume and revenue move together because average order values are close across branches (KES 68,353 to KES 75,591), so no branch wins on revenue through pricier baskets alone. Revenue shares: Nairobi 36.4%, Mombasa 19.5%, Nakuru 16.7%, Kisumu 15.1%, Eldoret 12.3%. **So what:** Nairobi alone brings in 36.4%, so keeping its stock and staffing right protects the biggest slice; the other branches are far smaller (KES 8,844,200 to KES 14,003,700), so growth effort is best aimed at lifting Eldoret, the smallest at 12.3%.

## Task 4 · Top three products

In [4]:
p = df.groupby("product").agg(revenue=("revenue","sum"), units_sold=("quantity","sum"))
print("By revenue:"); display(p.nlargest(3,"revenue"))
print("By units sold:");   display(p.nlargest(3,"units_sold"))

By revenue:


,revenue,units_sold
product,,
Laptop,41535000,639
Tablet,19348000,691
Phone,8880000,592


By units sold:


,revenue,units_sold
product,,
Tablet,19348000,691
Laptop,41535000,639
Phone,8880000,592


**Answer.** Top by revenue: Laptop, Tablet, Phone. Top by units: Tablet, Laptop, Phone. The same three products appear in both lists but in a different order: **Tablet** sells the most units (691) while **Laptop** earns the most revenue (KES 41,535,000) because revenue = units × price and Laptop costs KES 65,000 against KES 28,000 for Tablet. **Laptop** is 57.9% of revenue from only 21.2% of units, while **Tablet** is 22.9% of units but 26.9% of revenue; the top three products together make 97.2% of revenue. **So what:** a stock-out of Laptop costs the most money, so it deserves the tightest reorder level, while Tablet keeps volume moving.

## Task 5 · Weekend vs weekday

In [5]:
w = df.groupby("is_weekend").agg(orders=("order_id","count"), total_revenue=("revenue","sum"), avg_order_revenue=("revenue","mean"))
w.index = w.index.map({True:"Weekend", False:"Weekday"}); w

,orders,total_revenue,avg_order_revenue
is_weekend,,,
Weekday,723,51856200,"71,723.65"
Weekend,277,19936300,"71,972.20"


**Answer.** Weekdays: 723 orders, KES 51,856,200, avg KES 71,724. Weekends: 277 orders, KES 19,936,300, avg KES 71,972. Weekends cover only 2 of 7 days, so compare the average order value, not totals: average order values are virtually the same (+0.3%), and weekends hold 27.7% of orders against the 28.6% you would expect from 2 of 7 days, so there is no real weekend effect. **So what:** treat weekends like any other trading day; the data gives no reason for weekend-only promotions or extra weekend stock.

## Task 6 · Best and worst month

In [6]:
m = df.groupby(df.order_date.dt.month).revenue.sum()
m.index = [pd.Timestamp(2026,i,1).month_name() for i in m.index]
best, worst = m.idxmax(), m.idxmin()
print(m); print(f"best={best} worst={worst} difference={(m[best]-m[worst])/m[worst]*100:.1f}% (best vs worst)")

January     11786600
February    11772000
March       12396800
April       13518300
May         11415400
June        10903400
Name: revenue, dtype: int64
best=April worst=June difference=24.0% (best vs worst)


**Answer.** Best: **April** (KES 13,518,300); worst: **June** (KES 10,903,400). Best is **24.0%** higher than worst, i.e. worst is 19.3% below best. Caveat: the data ends on 2026-06-29, so June is missing its last day(s). **So what:** have stock and staff ready ahead of April demand, and find out what held June back (promotions, stock-outs or season) before planning the second half of the year.

## Task 7 · Premium online orders, Nairobi & Mombasa

In [7]:
sel = df[(df.price_band=="Premium") & (df.channel=="Online") & (df.branch.isin(["Nairobi","Mombasa"]))]
print(len(sel), "orders | revenue KES", f"{sel.revenue.sum():,.0f}")
sel.to_csv("outputs/premium_online_coast_capital.csv", index=False)

33 orders | revenue KES 5,850,000


**Answer.** **33** orders, total revenue **KES 5,850,000**. Exported to `outputs/premium_online_coast_capital.csv` (`index=False`). That is 8.1% of all revenue and 25.7% of online revenue (channel mix: In-store 57.1%, Online 31.7%, Phone order 11.2%); by branch: Nairobi KES 4,030,000, Mombasa KES 1,820,000. **So what:** Premium here means Laptop only, so this segment is a niche; growing online sales of that product in Nairobi and Mombasa is the lever, and the Phone order channel (11.2% of revenue) is too small to prioritise.

## Task 8 · Excel reconciliation (`excel/duka_branch_summary.xlsx`)

In [8]:
xl = pd.read_csv("excel/sumifs_recalc_check.csv").set_index("branch")      # SUMIFS/COUNTIFS values recalculated from the workbook
cmp = b[["total_revenue","orders"]].join(xl)
cmp["match"] = np.isclose(cmp.total_revenue, cmp.sumifs_revenue) & (cmp.orders == cmp.countifs_orders)
raw = pd.read_csv("excel/pivot_recalc_check.csv", header=None)         # values of the real PivotTable in the workbook
hdr_i = raw.index[raw.iloc[:,0].astype(str).str.contains("branch", case=False)][-1]
pv = raw.iloc[hdr_i+1:].copy(); pv.columns = ["branch"]+list(raw.iloc[hdr_i,1:]); pv = pv.set_index("branch")
pd_pivot = df.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum")
ok = np.allclose(pv.loc[pd_pivot.index, pd_pivot.columns].astype(float), pd_pivot)
tot = pv.loc[pv.index.str.contains("Total", case=False)].iloc[0].astype(float).iloc[-1]
print("SUMIFS = pandas for every branch:", cmp.match.all(), "| grand total:", f"{xl.sumifs_revenue.sum():,.0f} vs {df.revenue.sum():,.0f}")
print("PivotTable branch×channel = pandas pivot:", ok, "| pivot grand total:", f"{tot:,.0f}")
cmp

SUMIFS = pandas for every branch: True | grand total: 71,792,500 vs 71,792,500
PivotTable branch×channel = pandas pivot: True | pivot grand total: 71,792,500


,total_revenue,orders,sumifs_revenue,countifs_orders,match
branch,,,,,
Eldoret,8844200,117,8844200,117,True
Kisumu,10868100,159,10868100,159,True
Mombasa,14003700,186,14003700,186,True
Nairobi,26106100,376,26106100,376,True
Nakuru,11970400,162,11970400,162,True


**Answer.** The workbook has `Branch_Summary` (SUMIFS revenue + COUNTIFS orders + XLOOKUP region from the `Regions` sheet) and `Pivot_Branch_Channel` (a real PivotTable: branch rows × channel columns, sum of revenue). The check above shows every total equals pandas.